# 06 — Generate paper assets

**Purpose.** Convert verified canonical outputs into paper-ready tables and figures.

**Inputs.** Tables and predictions generated by Notebooks 01-05.

**Outputs.** Curated files in `outputs/paper/tables/` and `outputs/paper/figures/`.

**What you will learn.** How the paper assets are derived from saved results without retraining or retuning models.

**Run first.** Run Notebooks 01-05 first.

## Imports and paths

In [1]:
from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

current_path = Path.cwd().resolve()
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

assert project_root.name == "ml-finance-bankruptcy-analysis", (
    f"Unexpected project root: {project_root}"
)

DATA_DIR = project_root / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = project_root / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
REPORT_DIR = OUTPUTS_DIR / "report"
PAPER_TABLES_DIR = OUTPUTS_DIR / "paper" / "tables"
PAPER_FIGURES_DIR = OUTPUTS_DIR / "paper" / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORT_DIR,
    PAPER_TABLES_DIR,
    PAPER_FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


## Project constants

In [2]:
RAW_DATA_PATH = RAW_DATA_DIR / "american_bankruptcy.csv"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]

EXPECTED_MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "L1 Regularized Logistic Regression",
    "L2 Regularized Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
]

PREDICTION_TABLE_COLUMNS = [
    "model",
    COMPANY_COLUMN,
    YEAR_COLUMN,
    "actual_failed",
    "predicted_failed",
    "probability_failed",
]
PREDICTION_PROBABILITY_FLOAT_FORMAT = "%.11f"


## Feature names and descriptions

In [3]:
FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]


## Figure style helpers

In [4]:
MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Random Forest": "#59a14f",
    "Gradient Boosting": "#f28e2b",
}
OUTCOME_COLORS = {"detected": "#59a14f", "missed": "#e15759", "false_alarm": "#f28e2b"}
DIRECTION_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}
METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}
BASELINE_COLOR = "#8c8c8c"


def apply_project_style() -> None:
    """Apply the common Matplotlib style used across project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Add consistent grid and tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a Matplotlib figure with the project's standard export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Data validation and feature helpers

In [5]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Confirm that the raw dataset has the identifier, year, and target columns."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Confirm that the raw target contains only the expected alive/failed labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables from the raw dataset, excluding identifiers."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while keeping identifiers and target out of X."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictor matrix and binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()


## Company split helpers

In [6]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary


## Evaluation helpers

In [7]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the common classification metrics for one fitted model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def create_classification_report_table(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Convert scikit-learn's classification report into a flat table."""
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["alive", "failed"],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({"model": model_name, "class_or_average": label, **metrics})
    return pd.DataFrame(rows)


def create_prediction_table(
    data: pd.DataFrame,
    model_name: str,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> pd.DataFrame:
    """Create an identifier-preserving prediction table for one model."""
    table = pd.DataFrame(
        {
            "model": model_name,
            COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(),
            YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(),
            "actual_failed": data[TARGET_COLUMN].to_numpy(),
            "predicted_failed": y_pred,
            "probability_failed": probability_failed,
        }
    )
    return table[PREDICTION_TABLE_COLUMNS]


def save_prediction_table(predictions: pd.DataFrame, output_path: Path) -> None:
    """Write prediction probabilities with deterministic decimal formatting."""
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(
        output_path,
        index=False,
        float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT,
    )


## Model builders and selection helpers

In [8]:
def build_majority_class_baseline() -> DummyClassifier:
    """Build the benchmark model that always predicts the most frequent class."""
    return DummyClassifier(strategy="most_frequent")


def build_logistic_pipeline(
    penalty: str = "l2",
    c_value: float = 1.0,
    class_weight: str | dict | None = "balanced",
) -> Pipeline:
    """Build a scaled Logistic Regression pipeline with the original settings."""
    if penalty not in {"l1", "l2"}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")

    l1_ratio = 1.0 if penalty == "l1" else 0.0
    model = LogisticRegression(
        C=c_value,
        l1_ratio=l1_ratio,
        solver="saga",
        class_weight=class_weight,
        max_iter=5_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])


def build_interpretable_logit() -> Pipeline:
    """Build the fixed Logistic Regression benchmark used for interpretation."""
    return build_logistic_pipeline(penalty="l2", c_value=1.0)


def select_regularized_logit(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
    penalty: str,
    c_grid: list[float],
) -> tuple[Pipeline, float, float]:
    """Select L1 or L2 Logistic Regression by validation PR-AUC."""
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None or best_c is None:
        raise RuntimeError("No Logistic Regression model was selected.")
    return best_model, best_c, best_score


def build_decision_tree(max_depth: int | None = 3, min_samples_leaf: int = 100) -> DecisionTreeClassifier:
    """Build a class-weighted Decision Tree with the source project settings."""
    return DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def build_random_forest(
    n_estimators: int = 300,
    max_depth: int | None = 5,
    min_samples_leaf: int = 50,
    max_features: str | None = "sqrt",
) -> RandomForestClassifier:
    """Build a Random Forest using the original class weights and seed."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_gradient_boosting(
    n_estimators: int = 150,
    learning_rate: float = 0.05,
    max_depth: int = 2,
    min_samples_leaf: int = 100,
) -> GradientBoostingClassifier:
    """Build a Gradient Boosting classifier with the source project settings."""
    return GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )


def select_decision_tree(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[DecisionTreeClassifier, dict[str, int], float]:
    """Search the original Decision Tree grid and keep the best PR-AUC model."""
    grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid["max_depth"], grid["min_samples_leaf"]):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Decision Tree model was selected.")
    return best_model, best_params, best_score


def select_random_forest(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[RandomForestClassifier, dict[str, object], float]:
    """Search the original Random Forest grid and keep the best PR-AUC model."""
    grid = {
        "n_estimators": [300],
        "max_depth": [4, 6, None],
        "min_samples_leaf": [50, 100],
        "max_features": ["sqrt"],
    }
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(
        grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"], grid["max_features"]
    ):
        candidate = build_random_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
        )
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Random Forest model was selected.")
    return best_model, best_params, best_score


def select_gradient_boosting(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[GradientBoostingClassifier, dict[str, object], float]:
    """Search the original Gradient Boosting grid using balanced sample weights."""
    grid = {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [100],
    }
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
        grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["min_samples_leaf"]
    ):
        candidate = build_gradient_boosting(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Gradient Boosting model was selected.")
    return best_model, best_params, best_score


## Model reconstruction helpers

In [9]:
def parse_selected_parameters(value: object) -> dict[str, object]:
    """Parse the saved parameter string from `model_specification.csv`."""
    if pd.isna(value) or value == "":
        return {}

    text = str(value)
    if text.startswith("{"):
        return ast.literal_eval(text)

    params: dict[str, object] = {}
    for piece in text.split(","):
        if "=" in piece:
            key, val = piece.strip().split("=", 1)
            params[key] = float(val) if key == "C" else val
    return params


def build_model_from_spec(model_name: str, selected_parameters: object):
    """Recreate one model from its saved specification without using model binaries."""
    params = parse_selected_parameters(selected_parameters)
    if model_name == "Majority-class baseline":
        return build_majority_class_baseline()
    if model_name == "Logistic Regression":
        return build_interpretable_logit()
    if model_name == "L1 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l1", c_value=params["C"])
    if model_name == "L2 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l2", c_value=params["C"])
    if model_name == "Decision Tree":
        return build_decision_tree(**params)
    if model_name == "Random Forest":
        return build_random_forest(**params)
    if model_name == "Gradient Boosting":
        return build_gradient_boosting(**params)
    raise ValueError(f"Unknown model: {model_name}")


## Paper Table Functions and Outputs

In [10]:

PAPER_MODELS = ['Majority-class baseline', 'Logistic Regression', 'Random Forest', 'Gradient Boosting']
PAPER_CURVE_MODELS = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']
PAPER_METRICS = ['pr_auc', 'precision_failed', 'recall_failed', 'f1_failed']
PAPER_METRIC_LABELS = {'pr_auc': 'PR-AUC', 'precision_failed': 'Failed precision', 'recall_failed': 'Failed recall', 'f1_failed': 'Failed F1'}
PAPER_FINANCIAL_FEATURES = ['X8', 'X6', 'X11', 'X1', 'X17', 'X15']
target_distribution = pd.read_csv(TABLES_DIR / 'target_distribution.csv')
annual_failure_rate = pd.read_csv(TABLES_DIR / 'annual_failure_rate.csv')
class_feature_summary = pd.read_csv(TABLES_DIR / 'class_feature_summary.csv')
final_test_metrics = pd.read_csv(TABLES_DIR / 'final_test_metrics.csv')
final_test_predictions = pd.read_csv(TABLES_DIR / 'final_test_predictions.csv')
logistic_coefficients = pd.read_csv(TABLES_DIR / 'logistic_coefficients.csv')
tree_feature_importance = pd.read_csv(TABLES_DIR / 'tree_feature_importance.csv')
selected_thresholds = pd.read_csv(TABLES_DIR / 'selected_thresholds.csv')
pca_logistic_results = pd.read_csv(TABLES_DIR / 'pca_logistic_results.csv')

def round_columns(data, columns, decimals=3):
    out = data.copy(); out[columns] = out[columns].round(decimals); return out
paper_subset = final_test_metrics.set_index('model').loc[PAPER_MODELS].reset_index()
model_performance_summary = paper_subset.copy(); model_performance_summary.insert(0, 'evaluation_sample', 'Final test')
metric_columns = ['accuracy', 'balanced_accuracy', 'roc_auc', 'pr_auc', 'precision_failed', 'recall_failed', 'f1_failed']
model_performance_summary = round_columns(model_performance_summary[['evaluation_sample', 'model', *metric_columns]], metric_columns)
confusion_matrix_summary = paper_subset.rename(columns={'true_negative': 'correctly_identified_alive', 'false_positive': 'false_alarms', 'false_negative': 'missed_failures', 'true_positive': 'detected_failures'})
confusion_matrix_summary.insert(0, 'evaluation_sample', 'Final test')
confusion_matrix_summary = confusion_matrix_summary[['evaluation_sample', 'model', 'correctly_identified_alive', 'false_alarms', 'missed_failures', 'detected_failures', 'precision_failed', 'recall_failed']].rename(columns={'precision_failed': 'failed_precision', 'recall_failed': 'failed_recall'})
confusion_matrix_summary = round_columns(confusion_matrix_summary, ['failed_precision', 'failed_recall'])
medians = class_feature_summary.pivot(index=['feature', 'readable_name'], columns='status', values='median').reset_index().rename(columns={'alive': 'alive_median', 'failed': 'failed_median'})
medians['failed_minus_alive'] = medians['failed_median'] - medians['alive_median']
medians = medians.set_index('feature')
ordered_features = [feature for feature in PAPER_FINANCIAL_FEATURES if feature in medians.index]
remaining_features = [feature for feature in medians.index if feature not in ordered_features]
financial_median_summary = medians.loc[[*ordered_features, *remaining_features]].reset_index()[['feature', 'readable_name', 'alive_median', 'failed_median', 'failed_minus_alive']].round(2)
pca_summary = pca_logistic_results[['n_components', 'cumulative_explained_variance', 'roc_auc', 'pr_auc', 'precision_failed', 'recall_failed', 'f1_failed']].copy()
pca_summary.insert(0, 'evaluation_sample', 'Internal validation')
pca_summary = round_columns(pca_summary, ['cumulative_explained_variance', 'roc_auc', 'pr_auc', 'precision_failed', 'recall_failed', 'f1_failed'])
selected_threshold_summary = selected_thresholds.copy(); selected_threshold_summary.insert(0, 'evaluation_sample', 'Internal validation')
selected_threshold_summary = round_columns(selected_threshold_summary, ['threshold', 'precision_failed', 'recall_failed', 'f1_failed', 'predicted_failed_share'])
model_performance_summary.to_csv(PAPER_TABLES_DIR / 'model_performance_summary.csv', index=False, float_format='%.3f')
confusion_matrix_summary.to_csv(PAPER_TABLES_DIR / 'confusion_matrix_summary.csv', index=False, float_format='%.3f')
financial_median_summary.to_csv(PAPER_TABLES_DIR / 'financial_median_summary.csv', index=False)
pca_summary.to_csv(PAPER_TABLES_DIR / 'pca_summary.csv', index=False)
selected_threshold_summary.to_csv(PAPER_TABLES_DIR / 'selected_threshold_summary.csv', index=False)
model_performance_summary


,evaluation_sample,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed
0,Final test,Majority-class baseline,0.925,0.500,0.500,0.075,0.000,0.000,0.000
1,Final test,Logistic Regression,0.360,0.601,0.690,0.153,0.095,0.884,0.172
2,Final test,Random Forest,0.814,0.612,0.699,0.159,0.168,0.376,0.232
3,Final test,Gradient Boosting,0.679,0.638,0.688,0.158,0.132,0.590,0.216


## Paper Figures

In [11]:

apply_project_style()
plot_data = target_distribution.set_index('status_label').loc[['alive', 'failed']]
fig, ax = plt.subplots(figsize=(6.4, 4.0)); bars = ax.bar(plot_data.index, plot_data['count'], color=[MODEL_COLORS['Logistic Regression'], OUTCOME_COLORS['missed']])
ax.set_title('Class Distribution in the Raw Dataset'); ax.set_xlabel('Company status'); ax.set_ylabel('Number of company-year observations'); ax.set_ylim(0, plot_data['count'].max() * 1.12); style_axis(ax)
for bar, (_, row) in zip(bars, plot_data.iterrows(), strict=False):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + plot_data['count'].max()*0.015, f"{int(row['count']):,}\n({row['share']:.1%})", ha='center', va='bottom', fontsize=9)
save_figure(fig, PAPER_FIGURES_DIR / 'class_balance.png')
fig, ax = plt.subplots(figsize=(7.2, 4.1)); ax.plot(annual_failure_rate['year'], annual_failure_rate['failure_rate'], marker='o', color=MODEL_COLORS['Logistic Regression'])
ax.set_title('Observed Annual Failure Rate'); ax.set_xlabel('Year'); ax.set_ylabel('Failure rate'); ax.set_ylim(0, annual_failure_rate['failure_rate'].max() * 1.18); style_axis(ax, grid_axis='both', percent_y=True, integer_x=True); save_figure(fig, PAPER_FIGURES_DIR / 'annual_failure_rate.png')
plot_perf = model_performance_summary.set_index('model')[PAPER_METRICS].rename(columns=PAPER_METRIC_LABELS)
fig, ax = plt.subplots(figsize=(8.1, 4.6)); positions = list(range(len(plot_perf.columns))); offsets = [-0.18, -0.06, 0.06, 0.18]
for offset, (model_name, row) in zip(offsets, plot_perf.iterrows(), strict=False):
    ax.scatter([p + offset for p in positions], row.to_numpy(), marker='o', s=55, label=model_name, color=MODEL_COLORS.get(model_name, BASELINE_COLOR))
ax.set_title('Final-Test Ranking and Failed-Class Performance'); ax.set_xlabel('Metric'); ax.set_ylabel('Metric value'); ax.set_xticks(positions); ax.set_xticklabels(plot_perf.columns); ax.set_ylim(0, 1); style_axis(ax); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=4); save_figure(fig, PAPER_FIGURES_DIR / 'model_performance_summary.png')
plot_conf = confusion_matrix_summary.set_index('model')[['detected_failures', 'missed_failures', 'false_alarms']].rename(columns={'detected_failures': 'Failures detected', 'missed_failures': 'Failures missed', 'false_alarms': 'False alarms'})
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.5), width_ratios=[1.1, 1]); plot_conf[['Failures detected', 'Failures missed']].plot(kind='bar', ax=axes[0], color=[OUTCOME_COLORS['detected'], OUTCOME_COLORS['missed']]); plot_conf[['False alarms']].plot(kind='bar', ax=axes[1], color=[OUTCOME_COLORS['false_alarm']], legend=False)
for ax in axes:
    max_height = max(patch.get_height() for container in ax.containers for patch in container); ax.set_ylim(0, max_height * 1.12 if max_height else 1)
    for container in ax.containers: ax.bar_label(container, fmt='%.0f', padding=2, fontsize=8)
    ax.set_xlabel('Model'); ax.tick_params(axis='x', rotation=20); style_axis(ax)
axes[0].set_title('Failure detection'); axes[0].set_ylabel('Number of failed observations'); axes[0].legend(loc='upper right'); axes[1].set_title('Screening burden'); axes[1].set_ylabel('False alarms'); fig.suptitle('Final-Test Classification Outcomes', fontsize=12); save_figure(fig, PAPER_FIGURES_DIR / 'confusion_matrix_summary.png')
fig, ax = plt.subplots(figsize=(7.2, 4.9))
for model_name in PAPER_CURVE_MODELS:
    mp = final_test_predictions[final_test_predictions['model'] == model_name]
    precision, recall, _ = precision_recall_curve(mp['actual_failed'], mp['probability_failed'])
    ap = average_precision_score(mp['actual_failed'], mp['probability_failed'])
    ax.plot(recall, precision, color=MODEL_COLORS.get(model_name), linewidth=2.2, label=f'{model_name} (AP = {ap:.3f})')
base = final_test_predictions['actual_failed'].mean(); ax.axhline(base, linestyle='--', linewidth=1, color=BASELINE_COLOR, label=f'Prevalence baseline ({base:.1%})'); ax.set_title('Final-Test Precision-Recall Curves', pad=22); ax.set_xlabel('Recall for failed firms'); ax.set_ylabel('Precision for failed firms'); style_axis(ax, grid_axis='both'); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2); save_figure(fig, PAPER_FIGURES_DIR / 'precision_recall_key_models.png')
fig, ax = plt.subplots(figsize=(7.0, 4.8))
for model_name in PAPER_CURVE_MODELS:
    mp = final_test_predictions[final_test_predictions['model'] == model_name]
    fpr, tpr, _ = roc_curve(mp['actual_failed'], mp['probability_failed']); auc = roc_auc_score(mp['actual_failed'], mp['probability_failed'])
    ax.plot(fpr, tpr, color=MODEL_COLORS.get(model_name), linewidth=2.2, label=f'{model_name} (ROC-AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color=BASELINE_COLOR, label='Random classifier'); ax.set_title('Final-Test ROC Curves', pad=22); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); style_axis(ax, grid_axis='both'); ax.legend(loc='lower right'); save_figure(fig, PAPER_FIGURES_DIR / 'roc_curves_key_models.png')
plot_logit = logistic_coefficients.sort_values('absolute_coefficient', ascending=False).head(12).sort_values('coefficient')
fig, ax = plt.subplots(figsize=(7.3, 5.1)); bars = ax.barh(plot_logit['readable_name'], plot_logit['coefficient'], color=[DIRECTION_COLORS['positive'] if c > 0 else DIRECTION_COLORS['negative'] for c in plot_logit['coefficient']]); ax.axvline(0, linewidth=1, color='black'); ax.set_title('Largest Standardized Logistic Coefficients'); ax.set_xlabel('Logistic coefficient (standardized predictors)'); ax.set_ylabel('Financial variable'); style_axis(ax, grid_axis='x'); save_figure(fig, PAPER_FIGURES_DIR / 'top_logistic_coefficients.png')
models = ['Random Forest', 'Gradient Boosting']; top_features = tree_feature_importance[tree_feature_importance['model'].isin(models)].groupby(['feature', 'readable_name'], as_index=False)['importance'].max().sort_values('importance', ascending=False).head(8).sort_values('importance'); feature_order = top_features['readable_name'].tolist(); fig, axes = plt.subplots(1, 2, figsize=(10.4, 5.3), sharey=True)
for ax, model_name in zip(axes, models, strict=False):
    model_data = tree_feature_importance[tree_feature_importance['model'] == model_name].set_index('readable_name')
    plot_data = pd.DataFrame({'readable_name': feature_order, 'importance': [model_data.loc[feature, 'importance'] if feature in model_data.index else 0.0 for feature in feature_order]})
    ax.barh(plot_data['readable_name'], plot_data['importance'], color=MODEL_COLORS.get(model_name)); ax.set_title(model_name); ax.set_xlabel('Model-specific feature importance'); style_axis(ax, grid_axis='x')
fig.suptitle('Top Tree-Based Feature Importances\nImportance magnitudes are interpreted within each model.', fontsize=12); save_figure(fig, PAPER_FIGURES_DIR / 'top_tree_feature_importance.png')
fig, ax = plt.subplots(figsize=(6.9, 4.4)); ax.plot(pca_logistic_results['n_components'], pca_logistic_results['cumulative_explained_variance'], marker='o', color=MODEL_COLORS['Logistic Regression'])
for threshold in [0.8, 0.9, 0.95]: ax.axhline(threshold, linestyle='--', linewidth=1, color=BASELINE_COLOR, alpha=0.7)
ax.set_title('Explained Variance at Evaluated PCA Specifications'); ax.set_xlabel('Number of PCA components evaluated'); ax.set_ylabel('Cumulative explained variance'); ax.set_ylim(0, 1.05); style_axis(ax, grid_axis='both', percent_y=True, integer_x=True); save_figure(fig, PAPER_FIGURES_DIR / 'pca_explained_variance.png')
expected_tables = ['model_performance_summary.csv', 'confusion_matrix_summary.csv', 'financial_median_summary.csv', 'pca_summary.csv', 'selected_threshold_summary.csv']
expected_figures = ['class_balance.png', 'annual_failure_rate.png', 'model_performance_summary.png', 'confusion_matrix_summary.png', 'precision_recall_key_models.png', 'roc_curves_key_models.png', 'top_logistic_coefficients.png', 'top_tree_feature_importance.png', 'pca_explained_variance.png']
missing = [name for name in expected_tables if not (PAPER_TABLES_DIR / name).exists()] + [name for name in expected_figures if not (PAPER_FIGURES_DIR / name).exists()]
assert not missing, f'Missing paper assets: {missing}'
asset_checklist = pd.DataFrame({'asset': expected_tables + expected_figures, 'exists': True})
asset_checklist


,asset,exists
0,model_performance_summary.csv,True
1,confusion_matrix_summary.csv,True
2,financial_median_summary.csv,True
3,pca_summary.csv,True
4,selected_threshold_summary.csv,True
5,class_balance.png,True
6,annual_failure_rate.png,True
7,model_performance_summary.png,True
8,confusion_matrix_summary.png,True
9,precision_recall_key_models.png,True
